In [ ]:
import scipy.io as sio
import numpy as np
import matplotlib.pyplot as plt
import pysr 
from pysr import PySRRegressor
from scipy.optimize import least_squares
from sympy.utilities.codegen import codegen
from pathlib import Path

In [ ]:
import pandas as pd
df = pd.read_csv('EM_LP.csv', sep=';')
# Функция для проверки и преобразования строки в float (с заменой запятой на точку)
def convert_to_float(value):
    if type(value) == str:
        return float(value.replace(',', '.'))
    else:
        return value

for C in df.columns[:19]:
    df[C] = df[C].apply(lambda x: convert_to_float(x))

In [ ]:
df2 = pd.read_csv('EM_LP_2.csv', sep=';')
for C in df2.columns[:19]:
    df2[C] = df2[C].apply(lambda x: convert_to_float(x))

In [ ]:
import numpy as np

# Параметр задачи
mu = 1/82

# Координаты центров масс (в безразмерных единицах)
x1 = -mu    # Первое тело (например, Луна)
x2 = 1 - mu # Второе тело (например, Земля)

# Координаты L5 для CR3BP (равносторонний треугольник)
x_L5 = 0.5 - mu
y_L5 = np.sqrt(3)/2

def hessian_U(x, y, mu):
    r1 = np.sqrt((x + mu)**2 + y**2)
    r2 = np.sqrt((x - 1 + mu)**2 + y**2)
    
    dU_xx = 1 - (1-mu)*((y**2 - 2*(x+mu)**2)/r1**5) - mu*((y**2 - 2*(x-1+mu)**2)/r2**5)
    dU_yy = 1 - (1-mu)*(((x+mu)**2 - 2*y**2)/r1**5) - mu*(((x-1+mu)**2 - 2*y**2)/r2**5)
    dU_xy = 3*( (1-mu)*(x+mu)*y/r1**5 + mu*(x-1+mu)*y/r2**5 )
    
    return np.array([[dU_xx, dU_xy], [dU_xy, dU_yy]])

H = hessian_U(x_L5, y_L5, mu)
eigvals, eigvecs = np.linalg.eig(H)
a = 1/np.sqrt(np.abs(eigvals[0]))
b = 1/np.sqrt(np.abs(eigvals[1]))

print(f"Собственные значения: {eigvals}")
print(f"Полуоси эллипса Якоби: a = {a:.6f}, b = {b:.6f}")

In [ ]:
eigvecs

In [ ]:
0.49533458**2+0.86870228**2

In [ ]:
df.head()

In [ ]:
plt.plot(df['a'], df['x1'], label='x1')
plt.plot(df['a'], df['x2'], label='x2')
plt.grid()
plt.xlabel('a')
plt.ylabel('x')
plt.legend()
plt.show()

In [ ]:
plt.plot(df['a'], df['v1'], label='v1')
plt.plot(df['a'], df['v2'], label='v2')
plt.grid()
plt.legend()
plt.xlabel('a')
plt.ylabel('v')
plt.show()


In [ ]:
plt.plot(df2['a'], df2['v1'], label='v1')
plt.plot(df2['a'], df2['v2'], label='v2')
plt.grid()
plt.legend()
plt.xlabel('a')
plt.ylabel('v')
plt.show()


In [ ]:
a = df['a']
v1 = df['v1']
v2 = df['v2']

In [ ]:
mu = 1/82

# Вектор x для каждого угла (shape: (N, 2))
x = np.stack([np.cos(a + np.pi/3) - mu,
              -np.sin(a + np.pi/3)], axis=-1)

# Нормализация x (осевое деление)
x_norm = x / np.linalg.norm(x, axis=1, keepdims=True)

# Тангенциальный вектор для каждого x
tangent = np.stack([-x_norm[:,1], x_norm[:,0]], axis=-1)

In [ ]:
v = np.stack([v1, v2]).T

# Радиальная компонента (скаляр)
radial_comp = np.sum(v * x_norm, axis=1)
# Тангенциальная компонента (скаляр)
tangential_comp = np.sum(v * tangent, axis=1)

# Радиальная часть (вектор)
radial_proj = radial_comp[:,None] * x_norm
# Тангенциальная часть (вектор)
tangential_proj = tangential_comp[:,None] * tangent

print("Радиальные проекции shape:", radial_proj.shape)
print("Тангенциальные проекции shape:", tangential_proj.shape)

In [ ]:
# Нормы радиальных проекций (длина каждого радиального вектора)
radial_norms = np.linalg.norm(radial_proj, axis=1)

# Нормы тангенциальных проекций (длина каждого тангенциального вектора)
tangential_norms = np.linalg.norm(tangential_proj, axis=1)

In [ ]:
plt.plot(df['a'], radial_norms, label='rad')
plt.plot(df['a'], tangential_norms, label='tan')
plt.grid()
plt.legend()
plt.xlabel('a')
plt.ylabel('v')
plt.show()

In [ ]:
v.shape

In [ ]:
v_ell = eigvecs@v.T
plt.plot(df['a'], v_ell[0,:], label='ell1')
plt.plot(df['a'], v_ell[1,:], label='ell2')
plt.grid()
plt.legend()
plt.xlabel('a')
plt.ylabel('v')
plt.show()

In [ ]:
ax = plt.figure().add_subplot(projection='3d')
ax.scatter(df['a'][0:-1], df['Rho_1'][0:-1], df['Rho_1_i'][0:-1], label='Rho_1')
ax.scatter(df['a'][0:-1], df['Rho_2'][0:-1], df['Rho_2_i'][0:-1], label='Rho_2')
ax.scatter(df['a'][0:-1], df['Rho_3'][0:-1], df['Rho_3_i'][0:-1], label='Rho_3')
ax.scatter(df['a'][0:-1], df['Rho_4'][0:-1], df['Rho_4_i'][0:-1], label='Rho_4')
plt.grid()
plt.legend()
plt.xlabel('a')
plt.ylabel('Rho_4')
plt.show()

In [ ]:
df['a'][32]

In [ ]:
df['a'][68]

In [ ]:
plt.plot(df['a'][0:], df['T'][0:], label="T")
plt.scatter(df['a'][27], df['T'][27], label="1", )
plt.scatter(df['a'][32], df['T'][32], label="2", ) #эта
plt.scatter(df['a'][61], df['T'][61], label="3")
plt.scatter(df['a'][68], df['T'][68], label="4")   #эта тоже подозрительна
plt.grid()
plt.xlabel('a')
plt.legend()
plt.show()

In [ ]:
plt.plot(df['a'][0:], df['v2'][0:], label="v2")
plt.scatter(df['a'][27], df['v2'][27], label="1", )
plt.scatter(df['a'][32], df['v2'][32], label="2", )
plt.scatter(df['a'][61], df['v2'][61], label="3")
plt.scatter(df['a'][68], df['v2'][68], label="4")
plt.grid()
plt.xlabel('a')
plt.legend()
plt.show()

In [ ]:
#по мнению Суслова
plt.plot(df['a'][0:], df['T'][0:], label="T")
plt.scatter(df['a'][42], df['T'][42], label="1", )
plt.scatter(df['a'][47], df['T'][47], label="2", ) #эта
plt.scatter(df['a'][53], df['T'][53], label="3")
plt.scatter(df['a'][140], df['T'][140], label="4")   #эта тоже подозрительна
plt.grid()
plt.xlabel('a')
plt.legend()
plt.show()

In [ ]:
df['a'][53]

In [ ]:
plt.plot(df['a'][0:], df['v2'][0:], label="v2")
plt.scatter(df['a'][42], df['v2'][42], label="1", )
plt.scatter(df['a'][47], df['v2'][47], label="2", )
plt.scatter(df['a'][53], df['v2'][53], label="3")
plt.scatter(df['a'][140], df['v2'][140], label="4")
plt.grid()
plt.xlabel('a')
plt.legend()
plt.show()

In [ ]:
zero_crossings_first

In [ ]:
a = df['a'].values
v2 = df['v2'].values

# Считаем первую и вторую производные по a
dv2_da = np.gradient(v2, a)
d2v2_da2 = np.gradient(dv2_da, a)

def find_zeros(x, y):
    """Находит x-координаты нулей функции y(x) методом линейной интерполяции."""
    sign_change = np.where(np.diff(np.sign(y)))[0]
    zeros = []
    for i in sign_change:
        x0, x1 = x[i], x[i+1]
        y0, y1 = y[i], y[i+1]
        # Линейная интерполяция для точного положения нуля
        zero_x = x0 - y0 * (x1 - x0) / (y1 - y0)
        zeros.append(zero_x)
    return np.array(zeros)

# Находим нули производных
zeros_first = find_zeros(a, dv2_da)
zeros_second = find_zeros(a, d2v2_da2)

# Для отображения на графике получаем соответствующие значения v2 в этих точках
v2_at_zeros_first = np.interp(zeros_first, a, v2)
v2_at_zeros_second = np.interp(zeros_second, a, v2)

# Строим график
plt.figure(figsize=(10, 6))
plt.plot(a, v2, label="v2", color='blue')
plt.scatter(zeros_first, v2_at_zeros_first, color='red', marker='o', label="Нули 1-й производной")
plt.scatter(zeros_second, v2_at_zeros_second, color='green', marker='x', label="Нули 2-й производной")
plt.xlabel('a')
plt.ylabel('v2')
plt.legend()
plt.title("График функции v2 и нули её производных")
plt.grid(True)
plt.show()

In [ ]:
a = df['a'].values
v1 = df['v1'].values

# Считаем первую и вторую производные по a
dv1_da = np.gradient(v1, a)
d2v1_da2 = np.gradient(dv1_da, a)

# Находим нули производных
zeros_first = find_zeros(a, dv1_da)
zeros_second = find_zeros(a, d2v1_da2)

# Для отображения на графике получаем соответствующие значения v2 в этих точках
v1_at_zeros_first = np.interp(zeros_first, a, v1)
v1_at_zeros_second = np.interp(zeros_second, a, v1)

# Строим график
plt.figure(figsize=(10, 6))
plt.plot(a, v2, label="v1", color='blue')
plt.scatter(zeros_first, v1_at_zeros_first, color='red', marker='o', label="Нули 1-й производной")
plt.scatter(zeros_second, v1_at_zeros_second, color='green', marker='x', label="Нули 2-й производной")
plt.xlabel('a')
plt.ylabel('v1')
plt.legend()
plt.title("График функции v1 и нули её производных")
plt.grid(True)
plt.show()

In [ ]:
a = df['a'].values
T = df['T'].values

# Считаем первую и вторую производные по a
dT_da = np.gradient(T, a)
d2T_da2 = np.gradient(dT_da, a)


# Находим нули производных
zeros_first_T = find_zeros(a, dT_da)
zeros_second_T = find_zeros(a, d2T_da2)

# Для отображения на графике получаем соответствующие значения v2 в этих точках
T_at_zeros_first = np.interp(zeros_first_T, a, T)
T_at_zeros_second = np.interp(zeros_second_T, a, T)

# Строим график
plt.figure(figsize=(10, 6))
plt.plot(a, T, label="T", color='blue')
plt.scatter(zeros_first_T, T_at_zeros_first, color='red', marker='o', label="Нули 1-й производной")
plt.scatter(zeros_second_T, T_at_zeros_second, color='green', marker='x', label="Нули 2-й производной")
plt.xlabel('a')
plt.ylabel('T')
plt.legend()
plt.title("График функции T и нули её производных")
plt.grid(True)
plt.show()

In [ ]:
# Строим график
plt.figure(figsize=(10, 6))
sc = np.max(T)/np.max(v2)
plt.plot(a, T/sc, label="T", color='blue')
plt.scatter(zeros_first_T, T_at_zeros_first/sc, color='red', marker='o', label="Нули 1-й производной")
plt.scatter(zeros_second_T, T_at_zeros_second/sc, color='green', marker='x', label="Нули 2-й производной")
plt.plot(a, v2, label="v2", color='blue')
plt.scatter(zeros_first, v2_at_zeros_first, color='red', marker='o', label="Нули 1-й производной")
plt.scatter(zeros_second, v2_at_zeros_second, color='green', marker='x', label="Нули 2-й производной")
plt.xlabel('a')
plt.ylabel('v2')
plt.legend()
plt.title("График функции v2, T и нули её производных")
plt.grid(True)
plt.show()


In [ ]:
print(zeros_first_T)
print(zeros_second_T)
print(zeros_first)
print(zeros_second)

In [ ]:
print(a[24])
print(a[84])

In [ ]:
plt.plot(df['a'][0:], df['Jacobi Const'][0:], label="Jacobi Const")
plt.scatter(df['a'][27], df['Jacobi Const'][27], label="1", )
plt.scatter(df['a'][32], df['Jacobi Const'][32], label="2", )
plt.scatter(df['a'][61], df['Jacobi Const'][61], label="3")
plt.scatter(df['a'][68], df['Jacobi Const'][68], label="4")
plt.grid()
plt.xlabel('a')
plt.legend()
plt.show()

In [ ]:
#plt.plot(df['a'][0:], df['T'][0:]/np.max(df['T'][0:]), label="T")
#plt.plot(df['a'][0:], df['Rho_1'][0:]/np.max(df['Rho_1'][0:]), label="Rho_1", zorder=1)
plt.plot(df['a'][0:], df['Rho_1_i'][0:]/np.max(df['Rho_1_i'][0:]), label="Rho_1_i", zorder=1)
#plt.plot(df['a'][0:], df['Rho_2'][0:]/np.max(df['Rho_2'][0:]), label="Rho_2")
plt.plot(df['a'][0:], df['Rho_2_i'][0:]/np.max(df['Rho_2_i'][0:]), label="Rho_2_i", zorder=1)
#plt.plot(df['a'][0:], df['Rho_3'][0:]/np.max(df['Rho_3'][0:]), label="Rho_3")
plt.plot(df['a'][0:], df['Rho_3_i'][0:]/np.max(df['Rho_3_i'][0:]), label="Rho_3_i", zorder=1)
plt.plot(df['a'][0:], df['Rho_4'][0:]/np.max(df['Rho_4'][0:]), label="Rho_4", zorder=1)
plt.plot(df['a'][0:], df['Rho_4_i'][0:]/np.max(df['Rho_4_i'][0:]), label="Rho_4_i", zorder=1)
plt.scatter(df['a'][27], df['Rho_1_i'][27]/np.max(df['Rho_1_i'][0:]), label="1", zorder=2)
plt.scatter(df['a'][32], df['Rho_1_i'][32]/np.max(df['Rho_1_i'][0:]), label="2", zorder=2)
plt.scatter(df['a'][61], df['Rho_4'][61]/np.max(df['Rho_4'][0:]), label="3", zorder=2)
plt.scatter(df['a'][68], df['Rho_4'][68]/np.max(df['Rho_4'][0:]), label="4", zorder=2)
plt.grid()
plt.xlabel('a')
plt.legend()
plt.show()

In [ ]:
np.argmax(df['Jacobi Const'])

In [ ]:
print(df['a'][24])
print(df['a'][53])
print(df['a'][84])
print(df['a'][139])

In [ ]:
model = PySRRegressor(
        niterations=1,  # < Increase me for better results
        binary_operators=["*", "+", "-","/","^"],
        unary_operators=[
            'exp',
            'cos',
            'sin',
            'log',
        ],
        #extra_sympy_mappings={"pimult": lambda x: np.pi*x},
        #nested_constraints={"cos": {"cos": 0, "sin": 0},"sin": {"cos": 0, "sin": 0}},
        constraints={'^': (-1, 1)},
        #nested_constraints={"exp": {"exp": 0}},
        #nested_constraints={"exp": {"exp": 0},"log": {"log": 0}},
        #nested_constraints={"^": {"^": 0},"log": {"log": 0},"cos":{"cos": 0,"sin": 0,"log": 0},"sin":{"cos": 0,"sin": 0,"log": 0}},
        nested_constraints={"exp": {"exp": 0}, "^": {"^": 0},"log": {"log": 0},"cos":{"cos": 0,"sin": 0, "exp": 0,"log": 0},"sin":{"cos": 0,"sin": 0,"exp": 0,"log": 0}},
        populations=36,
        verbosity=1,
        maxsize=40,
        #batching=True,
        warm_start = False,
        #denoise=True
    )

In [ ]:
model = model.from_file(run_directory='outputs/20250727_143652_vN6BjZ/')

In [ ]:
model

In [ ]:
"(((log(a) ^ 2.5472412) + 0.03478206) / a) * (1.9898283 + (-0.0062812455 * (cos((a / -0.15525976) + -0.59622294) + ((exp(a ^ a) * -7.385333e-8) + (cos(a * 14.013148) * 0.04032319)))))"

In [ ]:
a=np.array(df['a'])
v1=np.array(df['v1'])
v2=np.array(df['v2'])
T=np.array(df['T'])

idxes = np.logical_and(a>1.66,a<2.805)
#idxes = a>2.74
a_part = a[idxes]
v1_part = v1[idxes]
v2_part = v2[idxes]
T_part = T[idxes]

input_values = np.expand_dims(a_part.flatten(),-1)
target_values = np.expand_dims(v1_part.flatten(),-1)

In [ ]:
from sympy.utilities.lambdify import lambdastr
from sympy.abc import a


# %%


N=20

test_preds = model.predict(input_values,N)
test_trues = target_values.reshape(test_preds.shape)
MSE = np.square(np.subtract(test_trues,test_preds)).mean() 
s = model.sympy(N).simplify()
display(s)
RMSE = np.sqrt(MSE)
print(RMSE)

print('Python Code:')
f = lambdastr(a, s)

print(f)

In [ ]:
#model.warm_start = True
#model.niterations=5000

In [ ]:
input_values = np.expand_dims(np.array(df['a'][53:85]).flatten(),-1)
target_values = np.expand_dims(np.array(df['v1'][53:85]).flatten(),-1)
model.fit(input_values, target_values,
        variable_names=["a"])

In [ ]:
from sympy.utilities.lambdify import lambdastr
from sympy.abc import a

In [ ]:
N=None

test_preds = model.predict(input_values,N)
test_trues = target_values.reshape(test_preds.shape)
MSE = np.square(np.subtract(test_trues,test_preds)).mean() 
s = model.sympy(N).simplify()
display(s)
RMSE = np.sqrt(MSE)
print(RMSE)

print('Python Code:')
f = lambdastr(a, s)
print(f)
plt.plot(input_values, test_trues)
plt.plot(input_values, test_preds,'--')
plt.grid()
plt.show()

In [ ]:
plt.plot(input_values, test_trues-test_preds)
plt.grid()
plt.show()

In [ ]:
np.min(df['a'])

In [ ]:
np.array(df['a'])

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
from scipy.integrate import solve_ivp
import math

def odeCR3BP(t, p, mu):
    R = np.sqrt((p[0] + mu)**2 + p[1]**2)
    r = np.sqrt((p[0] - 1 + mu)**2 + p[1]**2)
    dpdt = np.array([p[2], p[3],
    2 * p[3] + p[0] - (1 - mu) * (p[0] + mu) / R**3 - mu * (p[0] - 1 + mu) / r**3,
    -2 * p[2] + p[1] - (1 - mu) * p[1] / R**3 - mu * p[1] / r**3])

    return dpdt


mu = 1/82

x1_fun = lambda a: np.cos(a + np.pi/3) - mu
x2_fun = lambda a: -np.sin(a + np.pi/3)

def v1_fun(a):
    if a<0.5 and a>=0:
        x = ((-0.021469986*a - 0.0003111248*math.log(a) - 0.0014662967)*math.cos(4.2054663*a**2.4511335)) #1.1e-05
    elif a>=0.5 and a<1.08:
        x = ((0.02530322*(math.log(a) + 0.317646496301252)*math.sin(1.1552902/a) - 0.001310821*math.cos((a**2)**0.8824805))/math.cos((a**2)**0.8824805)) #2.4e-5
    elif a>=1.08 and a<1.7:
        #x = (-1.0*a + 2.26119382373287*math.log(a)**1.3880316 + 0.0014348506*math.sin(36.4078/a)/math.sin((a*(a - 0.5180937))**1.3899974) + 1.0332358) #1.0e-04 #(1.06,1.72)
        x = (a**a*(0.105399586*math.exp(a**(-16.428136)*math.sin(24.341038*a)/(math.sin(a**2)*math.sin(a**2.022637)) + 2.5595481*a**(-16.428136) - 8.104014*a**(-8.714068)) + 0.013663944)) #6.9e-05 #(1.06,1.72)
    elif a>=1.7 and a<2.8:
        #x = ((a + math.log(a))*(a*(-0.059331506*a + 1.63273748380587e-7*math.exp((a - 0.5849132)**a) + 0.34727088) - 0.3137545) + 0.038643904) #1.3e-4
        x = ((math.log(a)**2.5472412 + 0.03478206)*(4.63890896722515e-10*math.exp(a**a) - 0.000253279855733145*math.cos(14.013148*a) - 0.0062812455*math.cos(6.4408189217863*a + 0.59622294) + 1.9898283)/a) #3.5e-5
        #x = ((math.log(a)**2.5472412 + 0.03478206)*(- 0.000253279855733145*math.cos(14.013148*a) - 0.0062812455*math.cos(6.4408189217863*a + 0.59622294) + 1.9898283)/a) #3.5e-5 
    else:
        x = (1.1167183*math.log(a)*math.sin(a)**2/(math.sin(a**a)*math.sin(a**a - 0.0060529616)) + 0.950830622906542*math.log(a)*math.sin(a)/math.sin(a**a - 0.0060529616) - math.sin(a) + 1.3403602) #2.8e-05 #(2.76,2.81)
    return x
def v2_fun(a):
    if a<0.5 and a>=0:
        x =  (a*(0.057326578*a*math.cos(3.8822086*a - 2.3752067705845) + 0.0466879)) #7.3e-06
    elif a>=0.5 and a<1.08:
        x = (-1.67898636906521*(0.7340692*a)**a - 1.08062198721637*math.cos(a**a) + 1.87455570872575)#1.9e-4
    elif a>=1.08 and a<1.7:
        #x = (0.220146 - 0.17547299/(a*(a*(1.7710357 - 0.0043772506*math.cos(49.990993/a)/(a - 1.7396995)) + math.cos(2.23464336320977*a**2)))) #7.3e-5
        x = (0.04186942*a - 0.04186942*math.exp(math.cos(5.7263412*a) + 0.015981808*math.cos(51.677048/(a + 0.0395502))/(a*math.sin(5.726176/a))) + 0.04186942*math.log(math.sin(a**1.0287796)) + 0.13939385027384) #6.2e-05 #(1.08,1.7)
    elif a>=1.7 and a<2.8:
        x = (a*(0.31209984*a + (-0.07542186*a + 0.026630582*math.exp(a) + 0.56088626)*math.cos(a)) - 0.58147573) #5.8e-5
    else:
        x =  (-0.016114982*math.sin((a + 0.4091626)**a) - 0.19904901 + 1.9749417e-5/math.sin(2*a**2)) #2.7e-06 #(2.76,2.81)
    return x
def T_fun(a):
    if a<0.5 and a>=0:
        x =  (a**1.97235084*(-0.94786483*a + 0.798210717383035*a**a*math.cos(6.512141*a) + 1.13389117016559) + 21.027468) #6.9e-05
    elif a>=0.5 and a<1.08:
        x = (a + 0.2820783*math.exp(math.cos(4.1242228*a**1.7214167)) - math.cos(a) - math.cos(a*(a + 0.19541843)) + math.cos(4.0326896*a + 0.918334923647024) + 22.964064) #7.4e-3
    elif a>=1.08 and a<1.7:
        x = (1.0*(-0.065168686*math.sin(5.60516/a))**a*math.sin(50.968464*(a*(a + 0.6610624))**(-0.964687)) + 23.513718 + 1.0*math.exp(-math.sin(3.9780495*a**(-5.4722185) - 2.1772099))) #1.4e-03 #(1.08,1.7)
    elif a>=1.7 and a<2.8:
        #x = (1.0*math.log((a - 0.055466864)**(-1.4048711) - 2998199322.79278*math.exp(a)**(-16.559324)) - 2.06719275110082e-13*math.cos(a - 1.4642737)**(-16.55932) + 26.737839) #7.7e-4
        x = (1.0*math.log((a - 0.055466864)**(-1.4048711) - 2998199322.79278*math.exp(a)**(-16.559324)) + 26.737839) #7.7e-4
    else:
        x = (a*(19.13393 - 3.6059015*a) + 0.012532942*math.sin(919.7821*a)*math.sin(a*(a + 19.138603))) #4.7e-05 #(2.76,2.81)
    return x     
#v1 = lambda a: (-a*np.exp(-3.6822217*a**(-2.8971705) - 1.238104*a**3.7654412) + 0.046486337*a - 0.0993868964965362*a*np.exp(-a) + 0.29494765*a*np.exp(-3.6822217*a**(-2.8971705)))
#v2 = lambda a: (-0.02230581*(np.exp(a) + np.exp(np.cos((a**2)**0.95246226 - 1.3307008))*np.cos(3.6634598*a))*np.cos(2.0136378*a) + 0.052785624)
#T = lambda a: (-8.61150672305524*np.exp(-0.8734633*a**5.4633574 - 0.12579775409572*np.sin(a)*np.cos(a**5.97746) - 0.360278500850849*np.cos(a)*np.cos(a**5.97746)) + 1.0*np.cos(a**0.8264235) + 26.011023)

a_range = np.linspace(0.2, 2.81, 4)

fig_man, ax_man = plt.subplots(1)
for i in range(a_range.size):
    #print(i)
    p0 = np.array([x1_fun(a_range[i]), x2_fun(a_range[i]), v1_fun(a_range[i]), v2_fun(a_range[i])])
    p = solve_ivp(lambda t, p: odeCR3BP(t, p, mu),
                  t_span=(0, T_fun(a_range[i])),
                  y0=p0,
                  method='LSODA',
                  t_eval = np.linspace(0, T_fun(a_range[i]), 1000),
                  atol=1e-12,
                  rtol=1e-12)
    ax_man.plot(p.y[0,:], p.y[1,:], 'k')
    #ax_res.plot(x_range[i], np.linalg.norm(p.y[:,-1] - p0), '.k')
ax_man.plot(x1_fun(0), x2_fun(0),'*k')
ax_man.grid()
#ax_res.grid()

ax_man.set_xlabel('x, безразм.', fontsize=11)
ax_man.set_ylabel('y, безразм.', fontsize=11)
# Сохранение с высоким разрешением
figures_dir = Path('figures')
figures_dir.mkdir(exist_ok=True)
figname = 'L5_orbits.png'
plt.savefig(figures_dir / figname, dpi=300, bbox_inches='tight')


In [ ]:
#x_range = np.linspace(np.min(df['a']), np.max(df['a']), 300)
x_range = np.linspace(0.0001, 2.81, 300)
fig_res, ax_res = plt.subplots(1)
for i in range(x_range.size):
    #print(i)
    p0 = np.array([x1_fun(x_range[i]), x2_fun(x_range[i]), v1_fun(x_range[i]), v2_fun(x_range[i])])
    p = solve_ivp(lambda t, p: odeCR3BP(t, p, mu),
                  t_span=(0, T_fun(x_range[i])),
                  y0=p0,
                  method='LSODA',
                  t_eval = np.linspace(0, T_fun(x_range[i]), 1000),
                  atol=1e-12,
                  rtol=1e-12)
    ax_res.plot(x_range[i], np.linalg.norm(p.y[:,-1] - p0), '.k')
ax_res.grid()
ax_res.set_xlabel(r'$x$, безразм.', fontsize=11)
ax_res.set_ylabel(r'Невязка через период, безразм.', fontsize=11)
# Сохранение с высоким разрешением
figures_dir = Path('figures')
figures_dir.mkdir(exist_ok=True)
figname = 'L5_err.png'
plt.savefig(figures_dir / figname, dpi=300, bbox_inches='tight')

In [ ]:
#x_range = np.linspace(np.min(df['a']), np.max(df['a']), 300)
x_range = np.linspace(0.0001, 2.81, 300)
fig_res, ax_res = plt.subplots(1)
for i in range(x_range.size):
    #print(i)
    p0 = np.array([x1_fun(x_range[i]), x2_fun(x_range[i]), v1_fun(x_range[i]), v2_fun(x_range[i])])
    p = solve_ivp(lambda t, p: odeCR3BP(t, p, mu),
                  t_span=(0, T_fun(x_range[i])),
                  y0=p0,
                  method='LSODA',
                  t_eval = np.linspace(0, T_fun(x_range[i]), 1000),
                  atol=1e-12,
                  rtol=1e-12)
    ax_res.plot(x_range[i], np.linalg.norm(p.y[:,-1] - p0), '.k')
ax_res.grid()
ax_res.set_xlabel(r'$x$, безразм.', fontsize=11)
ax_res.set_ylabel(r'Невязка через период, безразм.', fontsize=11)
# Сохранение с высоким разрешением
figures_dir = Path('figures')
figures_dir.mkdir(exist_ok=True)
figname = 'L5_err.png'
plt.savefig(figures_dir / figname, dpi=300, bbox_inches='tight')

In [ ]:
a = np.array(df['a'])
trues = df['v1']
preds = [v1_fun(x) for x in df['a']]
preds = np.array(preds).flatten()

# Основной график
fig, ax = plt.subplots(figsize=(6,4))
ax.plot(a, trues, label='Истинное значение', lw=2)
ax.plot(a, preds, '--', label='Аппроксимация', lw=2)
ax.grid()
ax.set_xlabel(r'$a$, безразм.')
ax.set_ylabel(r'$v_x$, безразм.')
ax.legend(loc='lower right')

# Вложенный график (разница)
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
axins = inset_axes(ax, width="40%", height="30%", loc='upper center')
axins.plot(a, np.array(trues) - np.array(preds), color='black')
axins.set_title('Разница')
axins.grid()
axins.tick_params(labelsize=8)
axins.set_xlabel(r'$a$, безразм.', fontsize=8)
axins.set_ylabel('Δ, безразм.', fontsize=8)

# Сохранение с высоким разрешением
figures_dir = Path('figures')
figures_dir.mkdir(exist_ok=True)
figname = 'L5_vx.png'
plt.savefig(figures_dir / figname, dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
a = np.array(df['a'])
trues = df['v2']
preds = [v2_fun(x) for x in df['a']]
preds = np.array(preds).flatten()

# Основной график
fig, ax = plt.subplots(figsize=(6,4))
ax.plot(a, trues, label='Истинное значение', lw=2)
ax.plot(a, preds, '--', label='Аппроксимация', lw=2)
ax.grid()
ax.set_xlabel(r'$a$, безразм.')
ax.set_ylabel(r'$v_y$, безразм.')
ax.legend(loc='upper left')

# Вложенный график (разница)
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
axins = inset_axes(ax, width="40%", height="30%", loc='lower center')
axins.plot(a, np.array(trues) - np.array(preds), color='black')
axins.set_title('Разница')
axins.grid()
axins.tick_params(labelsize=8)
axins.set_xlabel(r'$a$, безразм.', fontsize=8)
axins.set_ylabel('Δ, безразм.', fontsize=8)

# Сохранение с высоким разрешением
figures_dir = Path('figures')
figures_dir.mkdir(exist_ok=True)
figname = 'L5_vy.png'
plt.savefig(figures_dir / figname, dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
a = np.array(df['a'])
trues = df['T']
preds = [T_fun(x) for x in df['a']]
preds = np.array(preds).flatten()

# Основной график
fig, ax = plt.subplots(figsize=(6,4))
ax.plot(a, trues, label='Истинное значение', lw=2)
ax.plot(a, preds, '--', label='Аппроксимация', lw=2)
ax.grid()
ax.set_xlabel(r'$a$, безразм.')
ax.set_ylabel(r'$T$, безразм.')
ax.legend(loc='upper left')

# Вложенный график (разница)
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
axins = inset_axes(ax, width="40%", height="30%", loc='center right')
axins.plot(a, np.array(trues) - np.array(preds), color='black')
axins.set_title('Разница')
axins.grid()
axins.tick_params(labelsize=8)
axins.set_xlabel(r'$a$, безразм.', fontsize=8)
axins.set_ylabel('Δ, безразм.', fontsize=8)

# Сохранение с высоким разрешением
figures_dir = Path('figures')
figures_dir.mkdir(exist_ok=True)
figname = 'L5_T.png'
plt.savefig(figures_dir / figname, dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
from fractions import Fraction
print(Fraction(21.027468/np.pi).limit_denominator(10))

In [ ]:
from ConvertFormula import expr_to_latex_with_consts
expr_str = "((-0.021469986*a - 0.0003111248*math.log(a) - 0.0014662967)*math.cos(4.2054663*a**2.4511335)) #1.1e-05"

litera = "B"
latex, consts = expr_to_latex_with_consts(expr_str, lhs="v_x", j=1, litera=litera)
#latex = latex.replace('x','\mu')
print(latex)
print("Константы:")
#for i, v in consts:
#    print(f"{litera}_{i}^{{1}} = {v}")
for i, v in consts:
    print(v)

In [ ]:
from ConvertFormula import expr_to_latex_with_consts
expr_str = "((0.02530322*(math.log(a) + 0.317646496301252)*math.sin(1.1552902/a) - 0.001310821*math.cos((a**2)**0.8824805))/math.cos((a**2)**0.8824805)) #2.4e-5"

litera = "B"
latex, consts = expr_to_latex_with_consts(expr_str, lhs="v_x", j=2, litera=litera)
#latex = latex.replace('x','\mu')
print(latex)
print("Константы:")
#for i, v in consts:
#    print(f"{litera}_{i}^{{1}} = {v}")
for i, v in consts:
    print(v)

In [ ]:
from ConvertFormula import expr_to_latex_with_consts
expr_str = "(a**a*(0.105399586*math.exp(a**(-16.428136)*math.sin(24.341038*a)/(math.sin(a**2)*math.sin(a**2.022637)) + 2.5595481*a**(-16.428136) - 8.104014*a**(-8.714068)) + 0.013663944)) #6.9e-05"

litera = "B"
latex, consts = expr_to_latex_with_consts(expr_str, lhs="v_x", j=3, litera=litera)
#latex = latex.replace('x','\mu')
print(latex)
print("Константы:")
#for i, v in consts:
#    print(f"{litera}_{i}^{{1}} = {v}")
for i, v in consts:
    print(v)

In [ ]:
from ConvertFormula import expr_to_latex_with_consts
expr_str = "((math.log(a)**2.5472412 + 0.03478206)*(4.63890896722515e-10*math.exp(a**a) - 0.000253279855733145*math.cos(14.013148*a) - 0.0062812455*math.cos(6.4408189217863*a + 0.59622294) + 1.9898283)/a)"

litera = "B"
latex, consts = expr_to_latex_with_consts(expr_str, lhs="v_x", j=4, litera=litera)
#latex = latex.replace('x','\mu')
print(latex)
print("Константы:")
#for i, v in consts:
#    print(f"{litera}_{i}^{{1}} = {v}")
for i, v in consts:
    print(v)

In [ ]:
from ConvertFormula import expr_to_latex_with_consts
expr_str = "(1.1167183*math.log(a)*math.sin(a)**2/(math.sin(a**a)*math.sin(a**a - 0.0060529616)) + 0.950830622906542*math.log(a)*math.sin(a)/math.sin(a**a - 0.0060529616) - math.sin(a) + 1.3403602)"

litera = "B"
latex, consts = expr_to_latex_with_consts(expr_str, lhs="v_x", j=5, litera=litera)
#latex = latex.replace('x','\mu')
print(latex)
print("Константы:")
#for i, v in consts:
#    print(f"{litera}_{i}^{{1}} = {v}")
for i, v in consts:
    print(v)

In [ ]:
from ConvertFormula import expr_to_latex_with_consts
expr_str = "(a*(0.057326578*a*math.cos(3.8822086*a - 2.3752067705845) + 0.0466879)) #7.3e-06"

litera = "С"
latex, consts = expr_to_latex_with_consts(expr_str, lhs="v_y", j=9, litera=litera)
#latex = latex.replace('x','\mu')
print(latex)
print("Константы:")
#for i, v in consts:
#    print(f"{litera}_{i}^{{1}} = {v}")
for i, v in consts:
    print(v)

In [ ]:
from ConvertFormula import expr_to_latex_with_consts
expr_str = "(-1.67898636906521*(0.7340692*a)**a - 1.08062198721637*math.cos(a**a) + 1.87455570872575)#1.9e-4"

litera = "С"
latex, consts = expr_to_latex_with_consts(expr_str, lhs="v_y", j=10, litera=litera)
#latex = latex.replace('x','\mu')
print(latex)
print("Константы:")
#for i, v in consts:
#    print(f"{litera}_{i}^{{1}} = {v}")
for i, v in consts:
    print(v)

In [ ]:
from ConvertFormula import expr_to_latex_with_consts
expr_str = "0.04186942*(a - math.exp(math.cos(5.7263412*a) + 0.015981808*math.cos(51.677048/(a+0.0395502))/(a*math.sin(5.726176/a))) + math.log(math.sin(a**1.0287796))) + 0.13939385027384 #6.2e-05"

litera = "С"
latex, consts = expr_to_latex_with_consts(expr_str, lhs="v_y", j=11, litera=litera)
#latex = latex.replace('x','\mu')
print(latex)
print("Константы:")
#for i, v in consts:
#    print(f"{litera}_{i}^{{1}} = {v}")
for i, v in consts:
    print(v)

In [ ]:
from ConvertFormula import expr_to_latex_with_consts
expr_str = "(a*(0.31209984*a + (-0.07542186*a + 0.026630582*math.exp(a) + 0.56088626)*math.cos(a)) - 0.58147573) #5.8e-5"

litera = "С"
latex, consts = expr_to_latex_with_consts(expr_str, lhs="v_y", j=12, litera=litera)
#latex = latex.replace('x','\mu')
print(latex)
print("Константы:")
#for i, v in consts:
#    print(f"{litera}_{i}^{{1}} = {v}")
for i, v in consts:
    print(v)

In [ ]:
from ConvertFormula import expr_to_latex_with_consts
expr_str = "(-0.016114982*math.sin((a + 0.4091626)**a) - 0.19904901 + 1.9749417e-5/math.sin(2*a**2)) #2.7e-06"

litera = "С"
latex, consts = expr_to_latex_with_consts(expr_str, lhs="v_y", j=13, litera=litera)
#latex = latex.replace('x','\mu')
print(latex)
print("Константы:")
#for i, v in consts:
#    print(f"{litera}_{i}^{{1}} = {v}")
for i, v in consts:
    print(v)

In [ ]:
from ConvertFormula import expr_to_latex_with_consts
expr_str = "(a**1.97235084*(-0.94786483*a + 0.798210717383035*a**a*math.cos(6.512141*a) + 1.13389117016559) + 21.027468) #6.9e-05"

litera = "D"
latex, consts = expr_to_latex_with_consts(expr_str, lhs="T", j=4, litera=litera)
#latex = latex.replace('x','\mu')
print(latex)
print("Константы:")
#for i, v in consts:
#    print(f"{litera}_{i}^{{1}} = {v}")
for i, v in consts:
    print(v)

In [ ]:
from ConvertFormula import expr_to_latex_with_consts
expr_str = "(a + 0.2820783*math.exp(math.cos(4.1242228*a**1.7214167)) - math.cos(a) - math.cos(a*(a + 0.19541843)) + math.cos(4.0326896*a + 0.918334923647024) + 22.964064) #7.4e-3"

litera = "D"
latex, consts = expr_to_latex_with_consts(expr_str, lhs="T", j=5, litera=litera)
#latex = latex.replace('x','\mu')
print(latex)
print("Константы:")
#for i, v in consts:
#    print(f"{litera}_{i}^{{1}} = {v}")
for i, v in consts:
    print(v)

In [ ]:
from ConvertFormula import expr_to_latex_with_consts
expr_str = "(-0.065168686*math.sin(5.60516/a)**a*math.sin(50.968464*(a*(a + 0.6610624))**(-0.964687)) + 23.513718 + 1.0*math.exp(-math.sin(3.9780495*a**(-5.4722185) - 2.1772099))) #1.4e-03"

litera = "D"
latex, consts = expr_to_latex_with_consts(expr_str, lhs="T", j=6, litera=litera)
#latex = latex.replace('x','\mu')
print(latex)
print("Константы:")
#for i, v in consts:
#    print(f"{litera}_{i}^{{1}} = {v}")
for i, v in consts:
    print(v)

In [ ]:
from ConvertFormula import expr_to_latex_with_consts
expr_str = "(math.log((a - 0.055466864)**(-1.4048711) - 2998199322.79278*math.exp(a)**(-16.559324)) + 26.737839) #7.7e-4"

litera = "D"
latex, consts = expr_to_latex_with_consts(expr_str, lhs="T", j=7, litera=litera)
#latex = latex.replace('x','\mu')
print(latex)
print("Константы:")
#for i, v in consts:
#    print(f"{litera}_{i}^{{1}} = {v}")
for i, v in consts:
    print(v)

In [ ]:
from ConvertFormula import expr_to_latex_with_consts
expr_str = "(a*(19.13393 - 3.6059015*a) + 0.012532942*math.sin(919.7821*a)*math.sin(a*(a + 19.138603))) #4.7e-05 #(2.76,2.81)"

litera = "D"
latex, consts = expr_to_latex_with_consts(expr_str, lhs="T", j=8, litera=litera)
#latex = latex.replace('x','\mu')
print(latex)
print("Константы:")
#for i, v in consts:
#    print(f"{litera}_{i}^{{1}} = {v}")
for i, v in consts:
    print(v)